In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score
import seaborn as sns

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16 #Change to 32 for cloud platforms

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # important for pretrained CNN
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

data_dir = "D:\Study\PVR_Lab\Transformer\chest_xray"

train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=val_test_transform)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes

In [ ]:
class LiteGatedTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        self.gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)

        gate = self.gate(x)
        x = gate * attn_output  # YOUR CORE CONTRIBUTION

        x = self.norm1(x + attn_output)

        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)

        return x

In [ ]:
class HybridLGT(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256):
        super().__init__()

        # 🔹 CNN Backbone (small change, big impact)
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])  # remove FC

        self.pool = nn.AdaptiveAvgPool2d((14, 14))

        self.flatten_dim = 512

        self.projection = nn.Linear(self.flatten_dim, embed_dim)

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.transformer = LiteGatedTransformerBlock(embed_dim, num_heads=4, ff_dim=512)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x)

        B, C, H, W = x.shape
        x = x.view(B, C, H*W).permute(0, 2, 1)  # tokens

        x = self.projection(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_out = x[:, 0]
        return self.classifier(cls_out)

In [ ]:
class LGT_NoCNN(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256):
        super().__init__()

        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=16, stride=16)

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.transformer = LiteGatedTransformerBlock(embed_dim, num_heads=4, ff_dim=512)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  # [B, C, H, W]

        B, C, H, W = x.shape
        x = x.flatten(2).permute(0, 2, 1)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_out = x[:, 0]
        return self.classifier(cls_out)

In [ ]:
class Transformer_NoGate(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)

        x = self.norm1(x + attn_output)

        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)

        return x

In [ ]:
class LGT_NoGate(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256):
        super().__init__()

        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])

        self.pool = nn.AdaptiveAvgPool2d((14, 14))

        self.projection = nn.Linear(512, embed_dim)

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.transformer = Transformer_NoGate(embed_dim, num_heads=4, ff_dim=512)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x)

        B, C, H, W = x.shape
        x = x.view(B, C, H*W).permute(0, 2, 1)

        x = self.projection(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_out = x[:, 0]
        return self.classifier(cls_out)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HybridLGT().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, epochs=10):
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # validation accuracy
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = correct / len(val_dataset)

        if val_acc > best_acc:
            best_acc = val_acc

        print(f"Epoch {epoch+1} - Val Acc: {val_acc:.4f}")

    return best_acc

In [ ]:
EPOCHS = 30
patience = 5

best_loss = float('inf')
counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []
val_f1s = []

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct = 0, 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_dataset)

    model.eval()
    val_loss, correct = 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(1)

            correct += (outputs.argmax(1) == labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = correct / len(val_dataset)

    # 🔥 F1 calculation
    val_f1 = f1_score(all_labels, all_preds)
    val_f1s.append(val_f1)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, F1={val_f1:.4f}")

    # Early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

y_true, y_pred, y_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)[:, 1]

        y_true.extend(labels.numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

print(classification_report(y_true, y_pred))

In [ ]:
#F1 curve plot
plt.plot(val_f1s, label="Validation F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("F1 Score Curve")
plt.legend()
plt.show()

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_probs)
auc = roc_auc_score(y_true, y_probs)

plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.legend()
plt.title("Accuracy Curve")
plt.show()

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.show()